<a href="https://colab.research.google.com/github/nonlinearbranch/ml-work/blob/main/optuna_ann_fashion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 import pandas as pd
 from sklearn.model_selection import train_test_split
 import torch
 from torch.utils.data import Dataset, DataLoader
 import torch.nn as nn
 import torch.optim as optim
 import matplotlib.pyplot as plt
 from torch.nn import Sequential


In [2]:
#for reproducability
torch.manual_seed(42)

In [3]:
#check gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
df = pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv', on_bad_lines='skip')
df.tail()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/fashion-mnist_train.csv'

# New Section

In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
X_train_tensor.shape

torch.Size([48000, 784])

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [ ]:
#dataset and dataloader

train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)



In [ ]:
import torch.nn as nn

#model/neural network class

class neural_network(nn.Module):

    def __init__(self, input_dim, num_hidden_layers, num_neurons_per_layer, output_dim, dropout_rate):
        super().__init__()
        layers = []
        current_input_dim = input_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(current_input_dim, num_neurons_per_layer))
            layers.append(nn.BatchNorm1d(num_neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            current_input_dim = num_neurons_per_layer

        layers.append(nn.Linear(current_input_dim, output_dim))

        self.network = nn.Sequential(*layers)

    def forward(self, X):
         out = self.network(X)
         return out

In [ ]:
#model class should be initialized priorly

#objective function
def objective_function(trial):
  #take next hyperparameter from search space
  num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
  num_neurons_per_layer = trial.suggest_int('num_neurons_per_layer', 8, 128, step = 8)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log = True)
  dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step = 0.1)
  batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log = True)

  #create dataloader
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory = True)
  test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory = True)
  #initialize model
  input_dim = 784
  output_dim = 10
  model = neural_network(input_dim, num_hidden_layers, num_neurons_per_layer, output_dim, dropout_rate)
  model = model.to(device)

  #learning parameters
  epochs = 30

  #optimizer
  if optimizer_name == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  elif optimizer_name == 'RMSprop':
    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  elif optimizer_name == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  #loss
  loss_fn = nn.CrossEntropyLoss()

  #model train loop( with deciding optimizer and loss fn)

  num_batches = len(train_loader)

  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        #move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        y_pred = model(batch_features.float())
        # Compute loss
        loss = loss_fn(y_pred, batch_labels)
        #gradient 0
        optimizer.zero_grad()

        #backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        #parameters update
        optimizer.step()

        total_epoch_loss += loss.item()

  #evaluate model and accuracy

  model.eval()

  with torch.no_grad():
     total = 0
     correct = 0
     #test accuracy evaluation
     for batch_features, batch_labels in test_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()
     accuracy = correct / total
  return accuracy





In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.9 MB/s eta 0:00:00


In [ ]:
import optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective_function, n_trials=10)

[I 2026-06-28 08:11:01,734] A new study created in memory with name: no-name-3e390214-7235-4419-892c-8f85f3df4889
[W 2026-06-28 08:11:18,491] Trial 0 failed with parameters: {'num_hidden_layers': 4, 'num_neurons_per_layer': 64, 'learning_rate': 0.05415352585942951, 'dropout_rate': 0.5, 'batch_size': 16, 'optimizer': 'SGD', 'weight_decay': 0.00023981386813348094} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_2228/2989540340.py", line 34, in objective_function
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/optim/sgd.py", line 65, in __init__
    super().__init__

KeyboardInterrupt: 

In [ ]:
study.best_value
study.best_params

In [ ]:
  #retrain
  #create dataloader
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory = True)
  test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory = True)

  #initialize model
  input_dim = 784
  output_dim = 10
  num_hidden_layers=5
  num_neurons_per_layer=112
  dropout_rate=0.4
  model2 = neural_network(input_dim, num_hidden_layers, num_neurons_per_layer, output_dim, dropout_rate)
  model2 = model2.to(device)

  #learning parameters
  epochs = 100
  learning_rate = 0.0003

  #optimizer
  optimizer = torch.optim.Adam(model2.parameters(), lr=learning_rate, weight_decay=6.019458450811453e-05)

  #loss
  loss_fn = nn.CrossEntropyLoss()

  #model train loop( with deciding optimizer and loss fn)

  num_batches = len(train_loader)

  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        #move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        y_pred = model2(batch_features.float())
        # Compute loss
        loss = loss_fn(y_pred, batch_labels)
        #gradient 0
        optimizer.zero_grad()

        #backward pass
        loss.backward()

        #parameters update
        optimizer.step()

        total_epoch_loss += loss.item()

    print(f"Epoch: {epoch+1}/{epochs}, Loss: {total_epoch_loss/num_batches}")


  #evaluate model and accuracy

  model2.eval()

  with torch.no_grad():
     total = 0
     correct = 0
     #test accuracy evaluation
     for batch_features, batch_labels in test_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model2(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     test_accuracy = correct / total
     print(f"test accuracy: {test_accuracy}")

     #training accuracy

     for batch_features, batch_labels in train_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model2(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     accuracy = correct / total
     print(f"test accuracy: {test_accuracy}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1/100, Loss: 1.2046472810705502
Epoch: 2/100, Loss: 0.7556325178543727
Epoch: 3/100, Loss: 0.6693029116590817
Epoch: 4/100, Loss: 0.6229488276143869
Epoch: 5/100, Loss: 0.5927996885081133
Epoch: 6/100, Loss: 0.5637699002921581
Epoch: 7/100, Loss: 0.5516888801157475
Epoch: 8/100, Loss: 0.5308333295981089
Epoch: 9/100, Loss: 0.523465081512928
Epoch: 10/100, Loss: 0.5146332148214181
Epoch: 11/100, Loss: 0.5056475806732973
Epoch: 12/100, Loss: 0.49955001326402027
Epoch: 13/100, Loss: 0.4934472989539305
Epoch: 14/100, Loss: 0.49062399689356484
Epoch: 15/100, Loss: 0.4758604375720024
Epoch: 16/100, Loss: 0.47286732933918635
Epoch: 17/100, Loss: 0.4668550878415505
Epoch: 18/100, Loss: 0.4627247002025445
Epoch: 19/100, Loss: 0.45433268311123054
Epoch: 20/100, Loss: 0.4501977316439152
Epoch: 21/100, Loss: 0.4496460625231266
Epoch: 22/100, Loss: 0.4457530233810345
Epoch: 23/100, Loss: 0.4426262631018956
Epoch: 24/100, Loss: 0.43767040194074314
Epoch: 25/100, Loss: 0.43401783511042596
Epoc

In [ ]:
model2.eval()

with torch.no_grad():
     total = 0
     correct = 0
     #test accuracy evaluation
     for batch_features, batch_labels in test_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model2(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     test_accuracy = correct / total
     print(f"test accuracy: {test_accuracy}")

     #training accuracy

     for batch_features, batch_labels in train_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model2(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     train_accuracy = correct / total
     print(f"train accuracy: {train_accuracy}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


test accuracy: 0.892
train accuracy: 0.9242166666666667
